In [ ]:
!pip install show langchain

**Installing the langchian module**

In [ ]:
!pip install -qU langchain

**creating a model to use in the creation of agent**

In [ ]:
!pip install -qU langchain-google-genai


In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

In [ ]:
gemini_api_key = userdata.get("GEMINI_API_KEY")

In [ ]:
model = init_chat_model(
    model="gemini-3.5-flash",
    model_provider="google_genai",
    api_key=gemini_api_key
)

**using tavily search engine for tool in ai agent**

In [ ]:
!pip install -qU langchain-tavily

In [ ]:
tavily_api_key = userdata.get("TAVILY_API_KEY")

In [ ]:
from langchain_tavily import TavilySearch
from  pprint import pprint

In [ ]:
skill_demand_tool = TavilySearch(
    max_results =5,
    search_depth = "advanced",
    topic = "general",
    tavily_api_key = tavily_api_key,
)

**Creating the job_search tool**

In [ ]:
rapidapi_key = userdata.get("RAPIDAPI_KEY")

In [ ]:
import requests
from langchain.tools import tool

In [ ]:
@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""

    print("\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")

    rapidapi_key = userdata.get('RAPIDAPI_KEY')

    url = "https://jsearch.p.rapidapi.com/search-v2"

    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }

    querystring = {
    "query": "software developer jobs in India",
    "country": "in",
    "num_pages": "1"
    }

    response = requests.get(
    url,
    headers=headers,
    params=querystring,
    timeout=30
    )

    print("Status Code:", response.status_code)

    response.raise_for_status()

    data = response.json()

    print("Complete API Response:")
    print(data)

    print("Response Keys:", data.keys())

    jobs_data = data.get("data", {})

    if isinstance(jobs_data, dict):
        jobs = jobs_data.get("jobs", [])
    elif isinstance(jobs_data, list):
        jobs = jobs_data
    else:
        jobs = []

    print("Extracted Jobs:", len(jobs))
    result = []

    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })

    return result

**importing the create_agent module form langchain.agents**

In [ ]:
from langchain.agents import create_agent

In [ ]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

**creating a agent by getting the model,tools,system_prompt**

In [ ]:
agent = create_agent(
    model = model,
    tools  = [skill_demand_tool, search_jobs],
    system_prompt = system_prompt,
    debug = True
)

In [ ]:
user_query = "What's the demand for generative ai in the industry and show me related job openings in India"

In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})

In [ ]:
print(response["messages"][-1].content[0]["text"])